# **From REST to reasoning: ingest, index, and query with dlt and Cognee.**

## Import and Installs

In [1]:
%%capture
!pip install cognee
!pip install kuzu

In [3]:
pip install -q "dlt[qdrant]" "qdrant-client[fastembed]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.9/100.9 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.0/329.0 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.8/324.8 kB 28.3 MB/s eta 0:00:00


#### QUESTION 1

In [5]:
pip show dlt

Name: dlt
Version: 1.12.3
Summary: dlt is an open-source python-first scalable data loading library that does not require any backend to run.
Home-page: https://github.com/dlt-hub
Author: 
Author-email: "dltHub Inc." <services@dlthub.com>
License: 
Location: /usr/local/lib/python3.11/dist-packages
Requires: click, fsspec, gitpython, giturlparse, hexbytes, humanize, jsonpath-ng, orjson, packaging, pathvalidate, pendulum, pluggy, pytz, pyyaml, requests, requirements-parser, rich-argparse, semver, setuptools, simplejson, sqlglot, tenacity, tomlkit, typing-extensions, tzdata
Required-by: cognee


Before running Cognee you have to specify up your environment.

Cognee relies on third-party LLM providers and you have a [great choice of them](https://docs.cognee.ai/how-to-guides/remote-models) you can use in your workflow.

**The simple way**

Just provide your OpenAI API key if you already have one. This will help you both with LLM and embeddings.

In [2]:
from google.colab import userdata
import os

os.environ["LLM_API_KEY"] = userdata.get('LLM_API_KEY') # it can be OpenAI API key
os.environ["GRAPH_DATABASE_PROVIDER"] = "kuzu"

## **Data we'll be using**

In this example, we’ll request data from an API that serves the **NYC taxi dataset**. For these purposes we created an API that can serve the data you are already familiar with.

### **API documentation**:  
- **Data**: Comes in pages of 1,000 records.  
- **Pagination**: When there’s no more data, the API returns an empty page.  
- **Details**:  
  - **Method**: GET  
  - **URL**: `https://us-central1-dlthub-analytics.cloudfunctions.net/data_engineering_zoomcamp_api`  
  - **Parameters**:  
    - `page`: Integer (page number), defaults to 1.  

Here’s how we design our requester:  
1. **Request page by page** until we hit an empty page. Since we don’t know how much data is behind the API, we must assume it could be as little as 1,000 records or as much as 10GB.
2. **Use a generator** to handle this efficiently and avoid loading all data into memory.  

The dates for taxi rides are all from within June 2009.

## **We'll be partitioning our data in our own way**

1. first_10_days
2. second_10_days
3. last_10_days

We'll be doing this manually for clarity, but dlt also supports partitioning, as you can find [here](https://dlthub.com/docs/plus/ecosystem/iceberg#partitioning).

#### QUESTION 2

In [ ]:
import dlt
import requests
import pandas as pd
from datetime import datetime
from dlt.destinations import qdrant

# Recurso DLT: coleta e prepara os dados
@dlt.resource(write_disposition="replace", name="zoomcamp_data")
def zoomcamp_data():
    url = "https://us-central1-dlthub-analytics.cloudfunctions.net/data_engineering_zoomcamp_api"
    response = requests.get(url)
    data = response.json()

    df = pd.DataFrame(data)
    df['Trip_Pickup_DateTime'] = pd.to_datetime(df['Trip_Pickup_DateTime'])

    # Criar faixas de datas para tag
    df['tag'] = pd.cut(
        df['Trip_Pickup_DateTime'],
        bins=[
            pd.Timestamp("2009-06-01"),
            pd.Timestamp("2009-06-10"),
            pd.Timestamp("2009-06-20"),
            pd.Timestamp("2009-06-30")
        ],
        labels=["first_10_days", "second_10_days", "last_10_days"],
        right=False
    )

    # Eliminar dados fora do range
    df = df[df['tag'].notnull()]
    yield df

# Configura o destino Qdrant
qdrant_destination = qdrant(qd_path="db.qdrant")

# Cria pipeline DLT
pipeline = dlt.pipeline(
    pipeline_name="zoomcamp_pipeline",
    destination=qdrant_destination,
    dataset_name="zoomcamp_tagged_data"
)

# Executa o pipeline
load_info = pipeline.run(zoomcamp_data())

# Mostra o log do último carregamento
print("\n✅ DLT Trace Info:")
print(pipeline.last_trace)

025-07-06 23:04:31,066|[WARNING]|858|137850685624320|dlt|worker.py|_get_items_normalizer:163|For data items in `zoomcamp_data` yielded as arrow and job file format jsonl native writer could not be found. A ArrowToJsonlWriter writer is used that internally converts arrow. This will degrade performance.
2025-07-06 23:04:31,110|[WARNING]|858|137850685624320|dlt|validate.py|verify_normalized_table:57|In schema `zoomcamp`: The following columns in table 'zoomcamp_data' did not receive any data during this load and therefore could not have their types inferred:
  - rate_code
  - mta_tax

Unless type hints are provided, these columns will not be materialized in the destination.
One way to provide type hints is to use the 'columns' argument in the '@dlt.resource' decorator.  For example:

@dlt.resource(columns={'rate_code': {'data_type': 'text'}})


✅ DLT Trace Info:
Run started at 2025-07-06 23:04:24.457481+00:00 and COMPLETED in 16.25 seconds with 4 steps.
Step extract COMPLETED in 1.73 seconds.

Load package 1751843069.3252559 is EXTRACTED and NOT YET LOADED to the destination and contains no failed jobs

Step normalize COMPLETED in 0.07 seconds.
Normalized data for the following tables:
- _dlt_pipeline_state: 1 row(s)
- zoomcamp_data: 998 row(s)

Load package 1751843069.3252559 is NORMALIZED and NOT YET LOADED to the destination and contains no failed jobs

Step load COMPLETED in 9.59 seconds.
Pipeline zoomcamp_pipeline load step completed in 9.56 seconds
1 load package(s) were loaded to destination qdrant and into dataset zoomcamp_tagged_data
The qdrant destination used /content/db.qdrant location to store data
Load package 1751843069.3252559 is LOADED and contains no failed jobs

Step run COMPLETED in 16.24 seconds.
Pipeline zoomcamp_pipeline load step completed in 9.56 seconds
1 load package(s) were loaded to destination qdrant and into dataset zoomcamp_tagged_data
The qdrant destination used /content/db.qdrant location to store data
Load package 1751843069.3252559 is LOADED and contains no failed jobs

#### QUESTION 3

In [7]:
import json

with open("db.qdrant/meta.json") as f:
    meta = json.load(f)

print(json.dumps(meta, indent=2))

{
  "collections": {
    "zoomcamp_tagged_data": {
      "vectors": {
        "fast-bge-small-en": {
          "size": 384,
          "distance": "Cosine",
          "hnsw_config": null,
          "quantization_config": null,
          "on_disk": null,
          "datatype": null,
          "multivector_config": null
        }
      },
      "shard_number": null,
      "sharding_method": null,
      "replication_factor": null,
      "write_consistency_factor": null,
      "on_disk_payload": null,
      "hnsw_config": null,
      "wal_config": null,
      "optimizers_config": null,
      "init_from": null,
      "quantization_config": null,
      "sparse_vectors": null,
      "strict_mode_config": null
    },
    "zoomcamp_tagged_data__dlt_pipeline_state": {
      "vectors": {
        "fast-bge-small-en": {
          "size": 384,
          "distance": "Cosine",
          "hnsw_config": null,
          "quantization_config": null,
          "on_disk": null,
          "datatype": null,
   